In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — TEST RÉSOLUTION SUR TOUS LES MATCHS DE RÉFÉRENCE, EN UN SEUL RUN (08/11/2026)
# Version 2 : validation KO en « medium », classification en « low », ESCALADE en résolution normale des cas où la
# validation KO et la classification ne sont pas d'accord (+ second regard en résolution normale).
# Nécessite pipeline.py et gemini_validator.py du 08/11 (2) sur GitHub.
# Rejoue test_resolution_gemini_cellule.py match par match (replays depuis les caches), SANS l'étape tenues/attribution
# (inutile pour vérifier les buts, ≈ 0,50 $ économisés par match), et dresse un bilan final : buts trouvés / vrais buts,
# faux buts, coût Gemini réel de chaque match.
# CRITÈRES FIXÉS AVANT : sur l'ensemble, AUCUN vrai but perdu par rapport aux références connues en résolution normale
# (Raeren 8/9 — penalty 69:12 jamais trouvé ; tous les autres : tous les buts) et au plus 1 faux but de plus par match.
# Coût prévu ≈ 1,50 $ par match (≈ 15 $ pour 10), durée ≈ 17 min par match (≈ 3 h). Save & Run All (Commit) conseillé.
# Entrées : codeprojectai (caches), les vidéos des 10 matchs, mziparameter, ET le fichier test_resolution_gemini_cellule.py
# (ajouté comme dataset, ou collé dans /kaggle/working/).
# ════════════════════════════════════════════════════════════════════════════════════════════
MATCHS_A_TESTER = ["spa", "minerois_ster", "franchimont", "raeren", "andrimont", "p1_minerois", "goe", "stembert",
                   "wanze", "juprelle"]
RESOLUTION = "medium"            # validation des coups d'envoi
RESOLUTION_CLASSIF = "low"       # classification des temps forts
REFERENCE = {   # (vrais buts, faux buts) : résolution NORMALE quand connue (spa, minerois_ster, franchimont, raeren),
    # sinon run « medium » seul du 08/11 (andrimont : cache arrêté à 61:39, 5 buts évaluables)
    "spa": (7, 0), "minerois_ster": (5, 1), "franchimont": (5, 4), "raeren": (8, 4), "andrimont": (5, 1),
    "p1_minerois": (3, 1), "goe": (2, 5), "stembert": (3, 3), "wanze": (2, 3), "juprelle": (1, 1)}

import glob, io, re, sys, time, contextlib

_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))
                 + glob.glob("/kaggle/working/test_resolution_gemini_cellule.py")), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (l'ajouter aux entrées ou dans /kaggle/working)")
CODE = open(_src, encoding="utf-8").read()
# on retire l'étape tenues + attribution (de « DÉTECTION DES TENUES » jusqu'au calcul du coût réel)
_a = CODE.index('print("\\n" + "=" * 60 + f"\\nDÉTECTION DES TENUES')
_b = CODE.index("# ── coût réel de ce run")
CODE = CODE[:_a] + "\n" + CODE[_b:]
CODE = re.sub(r'^RESOLUTION = .*$', f'RESOLUTION = "{RESOLUTION}"', CODE, count=1, flags=re.M)
CODE = CODE.replace('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION',
                    'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_KO"] = "{RESOLUTION}"\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_CLASSIF"] = "{RESOLUTION_CLASSIF}"\n'
                    'if "haute_resolution" not in open(_gv, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ gemini_validator.py du 08/11 (2) absent du dépôt GitHub")\n'
                    '_pp = next(iter(_g_res.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)), None)\n'
                    'if not _pp or "[ESCALADE]" not in open(_pp, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ pipeline.py du 08/11 (escalade) absent du dépôt GitHub")', 1)
assert "SCOUTIA_MEDIA_RESOLUTION_CLASSIF" in CODE, "point d'insertion introuvable"


import subprocess
# Chaque match tourne dans un PROCESSUS PYTHON SÉPARÉ : le cache Gemini, ses fichiers de sortie et les modules du
# pipeline repartent de zéro à chaque match (sinon l'état du 1er match se mélangerait aux suivants).
BILAN = {}
for m in MATCHS_A_TESTER:
    t0 = time.time()
    print(f"\n{'#' * 90}\n# MATCH {m} — résolution {RESOLUTION}\n{'#' * 90}", flush=True)
    code_m = re.sub(r'^MATCH = .*$', f'MATCH = "{m}"', CODE, count=1, flags=re.M)
    f = f"/kaggle/working/_test_resolution_{m}.py"
    open(f, "w", encoding="utf-8").write(code_m)
    proc = subprocess.Popen([sys.executable, "-u", f], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    lignes = []
    for l in proc.stdout:
        sys.stdout.write(l); lignes.append(l)
    proc.wait()
    sortie = "".join(lignes)
    b = re.search(r"BILAN DÉTECTION : (\d+)/(\d+) vrais buts \| (\d+) faux", sortie)
    c = re.search(r"TOTAL ≈ ([\d.]+) \$", sortie)
    manques = re.findall(r"but réel (\S+) : ❌ MANQUÉ", sortie)
    esc = re.search(r"\[ESCALADE\] (\d+) cas", sortie)
    changes = re.findall(r"\[ESCALADE\] (\S+) \(classification [^)]*\) : résolution réduite (\S+) → normale (\S+)", sortie)
    BILAN[m] = dict(trouves=int(b.group(1)) if b else None, vrais=int(b.group(2)) if b else None,
                    faux=int(b.group(3)) if b else None, cout=float(c.group(1)) if c else None,
                    manques=manques, minutes=(time.time() - t0) / 60, esc=int(esc.group(1)) if esc else 0,
                    changes=[f"{t} {a}→{b}" for t, a, b in changes if a != b])

print(f"\n{'═' * 90}\nBILAN — résolution {RESOLUTION} sur {len(BILAN)} match(s)\n{'═' * 90}")
print("   match           buts trouvés   faux   référence (trouvés, faux)   coût     durée   escalades  manqués")
tot_t = tot_v = tot_c = 0; perdus = []; trop_faux = []
for m, r in BILAN.items():
    ref = REFERENCE.get(m, (None, None))
    if r["trouves"] is None:
        print(f"   {m:15s} — résultat non lu (voir la sortie du match)"); continue
    tot_t += r["trouves"]; tot_v += r["vrais"]; tot_c += r["cout"] or 0
    if ref[0] is not None and r["trouves"] < ref[0]:
        perdus.append(m)
    if ref[0] is None and r["trouves"] < r["vrais"]:
        perdus.append(m + " (à vérifier : pas de référence récente)")
    if ref[1] is not None and r["faux"] > ref[1] + 1:
        trop_faux.append(m)
    print(f"   {m:15s} {r['trouves']:>3d} / {r['vrais']:<3d}       {r['faux']:>3d}    {str(ref):26s}  "
          f"{(r['cout'] or 0):5.2f} $  {r['minutes']:4.0f} min  {r['esc']:5d}      {', '.join(r['manques'])}"
          + (f"  | changés : {', '.join(r['changes'])}" if r['changes'] else ""))
n_esc = sum(r["esc"] for r in BILAN.values())
print(f"\n   TOTAL : {tot_t} / {tot_v} vrais buts | coût Gemini mesuré ≈ {tot_c:.2f} $ (sans l'attribution des équipes)")
print(f"   ⚠️ les {n_esc} escalades en résolution normale ont pu être relues GRATUITEMENT dans l'ancien cache : en production,"
      f" compter ≈ {n_esc * 0.017:.2f} $ de plus au total (≈ 0,017 $ par question KO en résolution normale)")
print(f"   matchs avec un vrai but de moins que la référence : {perdus or 'aucun'}")
print(f"   matchs avec plus d'1 faux but de plus que la référence : {trop_faux or 'aucun'}")
ok = not [p for p in perdus if 'à vérifier' not in p] and not trop_faux
print(f"   → {'RÉUSSI : la résolution « medium » peut devenir le réglage par défaut' if ok else 'ÉCHEC ou à examiner : voir les matchs listés'}")


##########################################################################################
# MATCH spa — résolution medium
##########################################################################################
  Clés lues dans /kaggle/input/datasets/michaelziant/mziparameter/param.env : SECRET_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, STRIPE_SECRET_KEY, STRIPE_PUBLIC_KEY, R2_SECRET_ACCESS_KEY, RUNPOD_API_KEY

ÉTAPE 1/3 — Clone
Cloning into '/kaggle/working/ProjectAIMika'...
Clone OK

ÉTAPE 2/3 — Dépendances
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━